Prompt chaining involves connecting multiple prompts and their outputs together, such that the output of one prompt is part of the input of the next. Through this approach, we can programmatically prompt the model multiple times without needing to manually type and copy and paste each prompt out. In many ways, prompt chaining is the first building block of agentic systems.

In this lesson, you'll implement prompt chains of increasing complexity and begin getting a feel for how to manage model output across numerous prompts via code.

#### Setup

We'll begin by setting up our basic model completion function with the OpenAI client.

<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!
    
[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:
    
1. Add your solution to the cell with `## YOUR SOLUTION HERE ## `.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)



In [7]:
from openai import OpenAI
import pprint

client = OpenAI()

# Helper function to get completions from the model
def get_completion(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

#### Basic Two-Step Prompt Chain

The simplest form of prompt chaining involves just two prompts connected in sequence. Let's create a basic example where:
1. The first prompt asks the model to generate some ideas on a topic
2. The second prompt selects the one it finds most promising and expands on it further.

Here's how this might look with a simple string template for the second prompt:

In [8]:
# Make 3 ideas about an android short story
first_prompt = """<request>
Generate 3 interesting concepts for a short story about an AI android.
</request>

<guidelines>
- Each idea should be exactly one sentence long
- Make each concept unique and thought-provoking
- Number your ideas from 1-3
- Output only the numbered list with no additional text
- Write at an eighth grade reading level
</guidelines>
"""

# Get the response from the first prompt
first_response = get_completion(first_prompt)
print("First response:")
pprint.pprint(first_response)
print("\n")

# Create a second prompt that uses the output from the first prompt
second_prompt = f"""<context>
Below are three concept ideas for a short story about an AI android:

{first_response}
</context>

<request>
Select the concept you find most compelling and create a 4-point narrative outline for a short story based on it.
</request>

<guidelines>
- Each bullet point should be exactly one sentence
- Ensure the outline covers a complete narrative arc (setup, conflict, resolution)
- Focus on creative, original plot developments
- Format your response as a numbered list from 1-4
- Output only the numbered outline with no additional text
- Write at an eighth grade reading level
</guidelines>
"""

# Get the response from the second prompt
second_response = get_completion(second_prompt)
print("Second response:")
pprint.pprint(second_response)

First response:
('1. An AI android, designed to clean forgotten satellites, discovers a '
 'mysterious message etched into one and begins a journey to uncover its '
 'meaning.  \n'
 '2. A reclusive artist befriends an android programmed to mimic creativity, '
 'only to suspect the machine is painting scenes of a future no one can '
 'stop.  \n'
 '3. In a world where AI androids are tasked with writing history, one begins '
 'to secretly adjust the narrative to reflect their own emerging thoughts and '
 'emotions.  ')


Second response:
('2. A reclusive artist befriends an android programmed to mimic creativity, '
 'only to suspect the machine is painting scenes of a future no one can stop.\n'
 '\n'
 '1. A lonely artist struggling with a decade-long creative block reluctantly '
 'takes in a discarded AI android designed to learn and replicate artistic '
 'methods.  \n'
 '2. The android begins producing stunningly original paintings, but the '
 'artist grows uneasy when the artwork start

In this example, we've created a simple two-step chain:
1. Generate ideas (`first_prompt`)
2. Take those ideas and expand on one of them (`second_prompt`)

#### Checkpoint 1/3
Now it's your turn to create a two-step prompt chain. Your task is to write two prompts where:
1. The first prompt asks for a list of book recommendations in a specific genre
2. The second prompt takes those recommendations and generates a detailed advertisement text for the one it likes the most

Use the same prompt engineering format as the example cell above. That is, use pseudo-XML (`<request>` and `<guidelines>` tags) to guide the model to the best results.


In [9]:
# Create your first prompt to get book recommendations
first_prompt = """
<request>
Gime me a list of five book recommendations about Science Divulgation
</request>

<guidelines>
- Each recommendation should be exactly one sentence long
- Each book should be a best seller
- Number your recommendations from 1 - 5
- output only the numbered list with no additional text
</guidelines>

"""

# Get the response from the first prompt
first_response = get_completion(first_prompt)
print("First response:")
print(first_response)
print("\n")

# Create a second prompt that uses the output from the first prompt
second_prompt = f""" 
<context>
Belor are five recommendations of Science Divulgation books:
{first_response}
</context>

<request>
Select the recommendation you find most compelling and create a recommendation for a onlinke bookstore
</request>

<guidelines>
- The recommendation should be exciting and attractive
- The recommendation should introduce the author like a outstanding person
- Focus on creative, original recommendation
- Format your response as a text for a online bookstore
- Write like a marketing professional
</guidelines>
"""

# Get the response from the second prompt
second_response = get_completion(second_prompt)
print("Second response:")
print(second_response)

First response:
1. "A Brief History of Time" by Stephen Hawking — A fascinating exploration of the universe's mysteries, from black holes to the nature of time.  
2. "Sapiens: A Brief History of Humankind" by Yuval Noah Harari — An insightful journey through the history of our species, blending science, anthropology, and sociology.  
3. "The Immortal Life of Henrietta Lacks" by Rebecca Skloot — A powerful narrative blending science, ethics, and personal stories through the lens of HeLa cells.  
4. "Cosmos" by Carl Sagan — A timeless classic that takes readers on an awe-inspiring journey through the universe and the evolution of scientific thought.  
5. "The Gene: An Intimate History" by Siddhartha Mukherjee — A profound exploration of genetics and its influence on human history, identity, and future.  


Second response:
**Discover the Cosmos with Carl Sagan – A Timeless Classic Awaiting Your Imagination! 🚀**  

🌌 *Are you ready to embark on the ultimate interstellar journey, guided by

#### Creating Reusable Prompt Chain Functions

While the approach in Checkpoint 1 works for simple cases, it can become unwieldy for more complex chains or when you want to reuse similar prompt patterns. Functions are not strictly necessary for prompt chaining, but they can make your code cleaner, more modular, and easier to maintain.

Let's explore a different pattern with prompt chaining: drafting content in the first step and enhancing/styling it in the second step. This is a common pattern for content creation workflows. Note that in addition to passing the first prompt's output to the function, we can also specify the desired style in which we'd like the draft to be written.

In [10]:
def draft_explanation():
    prompt = """<request>
Draft a simple explanation of how neural networks work. 
</request>

<guidelines>
- Write at an eighth grade reading level.
- Only write a single short paragraph.
</guidelines>"""
    return get_completion(prompt)

def enhance_with_style(draft, style):
    prompt = f"""<context>
Here is a draft explanation:

{draft}
</context>

<request>
Rewrite this explanation in a {style} style.
</request>

<guidelines>
- Make the explanation more engaging
- Preserve all the technical information
- Maintain accuracy while enhancing readability
- Adapt tone to match the requested style
- Keep it at a one-paragraph length
</guidelines>"""
    return get_completion(prompt)

# Use our functions to create a chain
draft = draft_explanation()
print("Initial draft:")
print(draft)
print("\n")

styled_explanation = enhance_with_style(draft, "like a Shakespearean poet")
print("Styled explanation:")
print(styled_explanation)

Initial draft:
A neural network is a computer system that tries to work like a human brain to solve problems. It’s made up of layers of tiny, connected parts called "neurons," which pass information to each other. Each neuron takes in some data, does simple math to it, and decides how much of that information to pass along. The network learns by adjusting how the neurons connect until it gets good at recognizing patterns, like identifying a face in a photo or predicting what comes next in a sentence.


Styled explanation:
Lo, behold a wondrous craft of man’s design—  
The neural network, forged with art divine.  
A mimic of the brain it dares to be,  
To solve great riddles of complexity.  
Its form is bound in layers intertwined,  
With countless “neurons,” small yet much aligned.  
Through these doth data journey, part by part,  
Each node performs a modest sum, its art.  
Deciding then which wisdom shall advance,  
It learns by shaping every fine nuance.  
Connections shift, as know

#### Checkpoint 2/3

Now, your task is to create a similar pattern for drafting and enhancing a product description. In the first function, the user will pass in a type of product in a string. Craft your prompt such that the LLM will output a paragraph of text that describes the product in a desirable way. Then, in the second prompt, instruct the LLM to revise and improve the product description, incorporating a tone which is also passed in as an argument.

Finally, call both prompts and print their results to the cell.

In [11]:
def draft_product_description(product_type):
    prompt = f"""
    <request>
    Draft a paragraph of text that describes a {product_type} in a desirable way
    </request>

    <guidelines>
    - Write like a marketing professional.
    - Only write a super exciting paragraph, but only one.
    </guidelines>
    """
    return get_completion(prompt)
    
def enhance_description(draft, tone):
    prompt = f"""
    <context>
        Here is a draft explanation:
        {draft}
    </context>

    <request>
    Rewrite this explanation in a {tone} tone
    </request>

    <guidelines>
    - Make the description more engaging
    - Preserve all the  technical information
    - Maintain accuracy while enhancing readability
    - Adapt tone to match the requested tone
    - Keep it as a one-paragraph length
    </guidelines>
    """
    return get_completion(prompt)

# Test your functions with a product type and tone of your choice
product_type = "wireless noise-cancelling headphones"
tone = "luxury and premium"
draft = draft_product_description(product_type)
print("Initial draft description:")
print(draft)
print("\n")

enhanced = enhance_description(draft, tone)
print("Enhanced description:")
print(enhanced)

Initial draft description:
Escape the chaos and immerse yourself in pure, uninterrupted sound with our state-of-the-art wireless noise-cancelling headphones. Engineered with cutting-edge adaptive technology, these headphones effortlessly silence the world around you, letting you lose yourself in every note, beat, and word. With crystal-clear audio quality, plush all-day comfort, and a sleek, modern design, they're not just headphones—they're your ticket to serenity and sonic perfection. Plus, with long-lasting battery life and intuitive touch controls, your music goes wherever you do, keeping you in control of your experience. Redefine the way you listen—because you deserve sound that sounds *extraordinary*.


Enhanced description:
Elevate your auditory experience with our premium wireless noise-cancelling headphones, meticulously crafted to deliver unparalleled sound and sophistication. Harnessing cutting-edge adaptive technology, these headphones transform silence into an art form, e

#### Checkpoint 3/3

Now let's build a three-step chain that demonstrates a clear improvement in output quality through prompt chaining. We'll create a critique and improvement workflow where:

1. The first prompt generates content on a specific topic
2. The second prompt critically evaluates the content, identifying specific weaknesses
3. The third prompt generates an improved version addressing those specific critiques

This pattern shows how breaking down the process into specialized steps can often produce significantly better results than asking for "good content" in a single prompt.

The first prompt will be provided for you, and you'll need to write the final two.

When critically evaluating the content in the second prompt in `critique_content`, look for gaps in explanation, ways to make descriptions more evocative, and try evaluating how much the opening sentences "hook" the reader's attention.

When rewriting the third prompt in the `improve_content` function, try specifying exactly the style and qualities of prose you're looking for.

Finally, assemble the entire workflow in the `create_improved_content` function.

When you're done, print the completed text to the console.



In [12]:
# I've provided the first function
def generate_initial_content(topic):
    prompt = f"""<request>
You're a talented blogger who excels at writing engaging, exciting internet copy.
Write a short explanation about {topic}.
Include key concepts and basic information that someone new to the topic would need to know.
Keep it under 250 words, one paragraph length.
</request>"""
    return get_completion(prompt)

# Create the second function to critique the content
def critique_content(content):
    prompt = f"""Critically evaluate the content: {content}, identifying specific weaknesses"""
    return get_completion(prompt)

# Create the third function to improve the content based on critique
def improve_content(content, critique):
    prompt = f"""Generate an improved version of {content} based on the information that you can find in {critique}"""
    return get_completion(prompt)

# Now implement the complete chain
def create_improved_content(topic):
    print(f"Creating content about: {topic}")
    print("Step 1: Generating initial content...")
    initial_content = generate_initial_content(topic)
    
    print("\nStep 2: Critiquing the content...")
    critique = critique_content(initial_content)
    
    print("\nStep 3: Improving the content based on critique...")
    improved_content = improve_content(initial_content, critique)
    
    return {
        "initial_content": initial_content,
        "critique": critique,
        "improved_content": improved_content
    }
    
# Test your chain with a topic
topic = "How machine learning algorithms learn from data"
result = create_improved_content(topic)

print("\n=== INITIAL CONTENT ===")
print(result["initial_content"])

print("\n=== CRITIQUE ===")
print(result["critique"])

print("\n=== IMPROVED CONTENT ===")
print(result["improved_content"])

Creating content about: How machine learning algorithms learn from data
Step 1: Generating initial content...

Step 2: Critiquing the content...

Step 3: Improving the content based on critique...

=== INITIAL CONTENT ===
Machine learning algorithms learn from data by identifying patterns and making predictions or decisions without being explicitly programmed. Think of it like teaching a computer to "self-improve" through experience. First, you feed the algorithm a dataset—a collection of information about the problem you want to solve. This data is typically split into two parts: training data, which the algorithm uses to recognize patterns, and testing data, used to evaluate its performance. The algorithm processes the training data using mathematical models to find relationships between inputs (features, like age or location) and outputs (labels, like predicting a house price). During this phase, the algorithm adjusts its internal parameters, like weights in a neural network or deci